# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [6]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [13]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy as part of a balanced diet.
    
    - Nutrients: good source of potassium, vitamin B6, vitamin C, and dietary fiber.
    - Benefits: supports heart health, digestion, and energy; low in fat.
    - Considerations: natural sugars rise as they ripen; portion size matters for blood sugar and calories.
    - Watch for: people with kidney disease may need to limit potassium; allergies are rare.
    - Tips: one medium banana (about 118 g) fits into a balanced snack or meal.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [8]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy and convenient. A medium banana (~105 calories) provides:

- Potassium ~ 420 mg
- Vitamin B6, vitamin C
- 3 g fiber, about 14 g sugar (natural)
- Some protein (~1 g)

Benefits: supports heart health, digestion, and steady energy; resistant starch when not fully ripe. Tips: pair with protein/fat for fullness; ripe bananas are sweeter (more sugar) but still part of a balanced diet. Consider kidney disease or very strict sugar limits as a caveat.

_Good Job!_